<a href="https://colab.research.google.com/github/Dhairya3103/movie-recommendation-website/blob/main/content_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import torch
from transformers import pipeline

In [ ]:
# Check if Colab GPU is active
device = 0 if torch.cuda.is_available() else -1
if device == 0:
    print("🚀 GPU is active! Text classification will be fast.")
else:
    print("⚠️ No GPU detected. Running on CPU (will be slower).")

In [ ]:
df = pd.read_csv('cleaned_movies_data.csv')
print(f"Loaded {len(df)} movies.")

Initialize the text model for deep learning - BART zero-shot

In [ ]:
# This model can classify text into any categories you invent!
text_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

In [ ]:
# 💡 Customize these themes to fit your project!
custom_themes = [
    "High-Octane Action & Explosions",
    "Slow-Burn Psychological Thriller",
    "Uplifting Underdog Story",
    "Sweeping Historical Epic",
    "Mind-Bending Sci-Fi",
    "Laugh-Out-Loud Comedy",
    "Dark & Gritty Crime",
    "Wholesome Family Adventure",
    "Heartbreaking Drama",
    "Spine-Chilling Horror"
]

Classification Function

In [ ]:
def extract_theme(overview_text, keywords_text):
    text_for_classification = ""
    # Only use keywords if available
    if pd.notna(keywords_text) and len(str(keywords_text)) > 0:
        text_for_classification = str(keywords_text)

    if text_for_classification:
        try:
            # Ask the AI to classify the text into one of our themes
            prediction = text_classifier(text_for_classification, candidate_labels=custom_themes)
            # Return the #1 most likely theme
            return prediction['labels'][0]
        except:
            return "Unclassified"
    else:
        return "No Keywords Available for Classification"

Run the Analysis - Batch Processing

In [ ]:
df_subset = df.copy()

themes = []

In [ ]:
# Loop through the movies
for index, row in df_subset.iterrows():
    theme = extract_theme(None, row['keywords']) # Pass None for overview, only use keywords
    themes.append(theme)

    # Print progress every 100 movies so you know it hasn't frozen
    if (index + 1) % 100 == 0:
        print(f"Processed {index + 1} movies...")

Testing

In [ ]:
df_subset['deep_theme'] = themes

In [ ]:
display(df_subset[['title', 'deep_theme', 'keywords']].head(10))

In [ ]:
movie_name = input("Enter a movie name to classify: ")

# Find the movie's overview and keywords in the DataFrame
selected_movie = df[df['title'].str.lower() == movie_name.lower()]

if not selected_movie.empty:
    keywords = selected_movie['keywords'].iloc[0] # Get keywords
    result_theme = extract_theme(None, keywords) # Pass None for overview, only use keywords
    print(f"The movie '{movie_name}' has the theme: {result_theme}")
else:
    print(f"Movie '{movie_name}' not found in the dataset or has no data to classify.")

Save the updated Dataset

In [ ]:
df_subset.to_csv('themed_movies_data.csv', index=False)